In [4]:
!pip install sacrebleu
!pip install transformers
!pip install accelerate
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.4 MB/s eta 0:00:00


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
data= "/content/drive/MyDrive/data"
src = os.path.join(data,"dev.de-hsb.de")
tgt =os.path.join(data,"dev.de-hsb.hsb")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import re
import torch
from tqdm import tqdm
import evaluate
if os.path.exists(src) and os.path.exists(tgt):
  with open(src,  encoding='utf-8') as f:
    val_s =[line.strip() for line in f.readlines() if line.strip()]
  with open(tgt,encoding='utf-8') as f:
    val_t = [line.strip() for line in f.readlines() if line.strip()]
  model_name = "Qwen/Qwen2.5-0.5B"
  tokenizer = AutoTokenizer.from_pretrained(model_name)
  model = AutoModelForCausalLM.from_pretrained(
      model_name,
      dtype=torch.bfloat16,
      device_map="auto",
  )
  if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
  tokenizer.padding_side = "left"
  if tokenizer.chat_template is None:
    tokenizer.chat_template = "{% if not add_generation_prompt is defined %}{% set add_generation_prompt = false %}{% endif %}{% for message in messages %}{{'<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>' + '\n'}}{% endfor %}{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %}"
  def extract_translation(text):
    match =   re.search(r"<hsb>\s*(.*?)\s*</hsb>",text, re.DOTALL)
    if match:
      return match.group(1).strip()
    return text.strip()
  predictions = []
  batch_size =16
  instruction_text = "Translate the following German text to Upper Sorbian.\nPut it in this format <hsb> Upper Sorbian translation </hsb>."
  for i in tqdm(range(0, len(val_s),batch_size)):
    batch_src= val_s[i:i+batch_size]
    batch_prompts = []
    for src in batch_src:
      messages = [
          {"role": "system", "content": instruction_text},
          {"role": "user", "content": f"<deu> {src} </deu>"}
      ]
      text= tokenizer.apply_chat_template(messages,tokenize=False, add_generation_prompt=True)
      batch_prompts.append(text)
    inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=True).to(model.device)
    with torch.no_grad():
      generated_ids = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id
      )
    generated_ids = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated_ids)]
    raw_preds =tokenizer.batch_decode(generated_ids,skip_special_tokens=True)
    clean_preds = [extract_translation(p) for p in raw_preds]
    predictions.extend(clean_preds)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

100%|██████████| 250/250 [2:16:54<00:00, 32.86s/it]


In [ ]:
sacrebleu = evaluate.load("sacrebleu")
chrf = evaluate.load("chrf")
bleu_result = sacrebleu.compute(predictions=predictions, references=[[r] for r in val_t])
chrf_result = chrf.compute(predictions=predictions, references=[[r] for r in val_t], word_order=2)
print(f"SacreBLEU: {bleu_result['score']:.2f}")
print(f"chrF++:    {chrf_result['score']:.2f}")

SacreBLEU: 0.06
chrF++:    4.24
